In [122]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
# parcel1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
# res='1km';t_res='5min'
# Np_str='1e6'

# # dx = 1km; Np = 50M
# #Importing Model Data
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# dx = 1km; Np = 50M; Nz = 95
#Importing Model Data
dir2='/home/air673/koa_scratch/'
data1=xr.open_dataset(dir2+'cm1out_1km_1min_95nz.nc') #***
parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_95nz.nc') #***
res='1km'; t_res='1min_95nz'; Np_str='50e6'

In [123]:
w_thresh1=0.1
w_thresh2=0.5
qcqi_thresh=1e-6

In [124]:
%%time

shape = data1['winterp'].shape  # (nx, ny, nz, nt)
chunk_sizes = (shape[0], shape[1], 10, 25)  # full x, full y, chunk z, chunk time

# Estimate total number of chunks (x and y are not chunked)
nz_chunks = (shape[2] + chunk_sizes[2] - 1) // chunk_sizes[2]
nt_chunks = (shape[3] + chunk_sizes[3] - 1) // chunk_sizes[3]
total_chunks = nz_chunks * nt_chunks

def chunk_z_t(shape, chunk_sizes):
    for ts in range(0, shape[3], chunk_sizes[3]):
        for zs in range(0, shape[2], chunk_sizes[2]):
            yield (
                slice(ts, min(ts + chunk_sizes[3], shape[3])),
                slice(zs, min(zs + chunk_sizes[2], shape[2]))
            )

lfc_profile=np.zeros((1,2))

# Loop with full x and y
for i, (ts, zs) in enumerate(chunk_z_t(shape, chunk_sizes), 1):
    if i % 1 == 0: print(f'Processing chunk {i}/{total_chunks}: z={zs}, t={ts}')

    lfc_chunk = data1['lfc'].isel(time=ts).data
    w_chunk = data1['winterp'].isel(zh=zs, time=ts).data     # shape: (x, y, z_chunk, t_chunk)
    q_chunk = (data1['qc'].isel(zh=zs, time=ts).data +
               data1['qi'].isel(zh=zs, time=ts)).data

    # (x, y, z, t) mask
    mask = (w_chunk > w_thresh2) & (q_chunk > qcqi_thresh)
    t,z,y,x=np.where(mask)
    unique_coords= GetUniqueCoords(t,y,x)
    lfc_list=lfc_chunk[t,y,x]; lfc_list=lfc_list[lfc_list>0]

    lfc_profile[:,0]+=np.sum(lfc_list)
    lfc_profile[:,1]+=len(lfc_list)
    

Processing chunk 1/420: z=slice(0, 10, None), t=slice(0, 25, None)
Processing chunk 2/420: z=slice(10, 20, None), t=slice(0, 25, None)
Processing chunk 3/420: z=slice(20, 30, None), t=slice(0, 25, None)
Processing chunk 4/420: z=slice(30, 40, None), t=slice(0, 25, None)
Processing chunk 5/420: z=slice(40, 50, None), t=slice(0, 25, None)
Processing chunk 6/420: z=slice(50, 60, None), t=slice(0, 25, None)
Processing chunk 7/420: z=slice(60, 70, None), t=slice(0, 25, None)
Processing chunk 8/420: z=slice(70, 80, None), t=slice(0, 25, None)
Processing chunk 9/420: z=slice(80, 90, None), t=slice(0, 25, None)
Processing chunk 10/420: z=slice(90, 100, None), t=slice(0, 25, None)
Processing chunk 11/420: z=slice(100, 110, None), t=slice(0, 25, None)
Processing chunk 12/420: z=slice(110, 120, None), t=slice(0, 25, None)
Processing chunk 13/420: z=slice(120, 130, None), t=slice(0, 25, None)
Processing chunk 14/420: z=slice(130, 140, None), t=slice(0, 25, None)
Processing chunk 15/420: z=slice(14

In [126]:
#TAKING MEAN
mean_LFC = lfc_profile[:,0]/lfc_profile[:,1]
print(mean_LFC)

#OUTPUTTING
import pickle
dir2=dir+f'Project_Algorithms/Domain_Profiles/'
output_file=dir2+f'mean_lfc_{res}_{t_res}_{Np_str}.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(mean_LFC, f)

[1917.60926758]


In [121]:
# #READING BACK IN
# import pickle
# dir2 = dir + f'Project_Algorithms/Domain_Profiles/'
# input_file = dir2 + f'mean_lfc_{res}_{t_res}_{Np_str}.pkl'

# with open(input_file, 'rb') as f:
#     mean_LFC_loaded = pickle.load(f)
# print(mean_LFC_loaded)